In [0]:
# Master Import Cell
from pyspark.sql.functions import col, when, udf, length
from pyspark.sql.types import FloatType
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression

# Confirming imports are loaded
print("All Spark functions and ML libraries imported successfully.")


In [0]:
# Install NLP library
%pip install textblob

# Restart Python to register the new library
dbutils.library.restartPython()

# Standard Imports
from pyspark.sql.functions import col, length, when, udf
from pyspark.sql.types import FloatType
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression
from textblob import TextBlob


In [0]:
# Define path to compressed JSONL file
path = "/Volumes/workspace/default/project/Toys_and_Games.jsonl.gz"

# Ingest data using Spark
df = spark.read.json(path)

# Data Understanding: Check schema and row count to justify "Big Data" methodology [cite: 17]
df.printSchema()
print(f"Total Records: {df.count()}")

# Show initial structure
display(df.limit(10))

In [0]:
from pyspark.sql.types import FloatType

In [0]:
# 1. Import the library
from textblob import TextBlob 
from pyspark.sql.functions import udf, col, when
from pyspark.sql.types import FloatType

# 2. Define Sentiment Function
def get_sentiment(text):
    if text is None:
        return 0.0
    # The worker now knows what TextBlob is because of the import above
    return TextBlob(text).sentiment.polarity

# 3. Register UDF
sentiment_udf = udf(get_sentiment, FloatType())

# 4. Apply and Categorize
df_sentiment = df.limit(1000).withColumn("sentiment_score", sentiment_udf(col("text")))

df_final = df_sentiment.withColumn("sentiment_label",
    when(col("sentiment_score") > 0.1, "Positive")
    .when(col("sentiment_score") < -0.1, "Negative")
    .otherwise("Neutral")
)

# 5. Display result
display(df_final.select("text", "sentiment_score", "sentiment_label").limit(20))

In [0]:
from pyspark.sql.functions import length

# 1. Create Features: Review length and Sentiment
df_model_prep = df_final.withColumn("review_length", length(col("text")))

# 2. Create Target (Label): 1 if rating is > 4 (Highly Rated), else 0
# Note: Check if your rating column is named 'overall' or 'rating'
df_model_prep = df_model_prep.withColumn("label", when(col("rating") > 4, 1).otherwise(0))

In [0]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=["sentiment_score", "review_length"],
    outputCol="features",
    handleInvalid="skip"
)

output = assembler.transform(df_model_prep)
train_data, test_data = output.randomSplit([0.7, 0.3], seed=42)


In [0]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# 1. Initialize the Model [cite: 13]
# We use the 'features' vector we created and the 'label' (rating > 4)
lr = LogisticRegression(featuresCol="features", labelCol="label")

# 2. Train the Model on 70% of the data
lr_model = lr.fit(train_data)

# 3. Make Predictions on the remaining 30%
predictions = lr_model.transform(test_data)

# 4. Evaluate Performance (Goal: F1-Score) [cite: 14]
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")
f1_score = evaluator.evaluate(predictions)

print(f"--- Model Evaluation Results ---")
print(f"F1-Score: {f1_score}")

# 5. Preview Predictions for your technical report [cite: 35]
display(predictions.select("text", "rating", "label", "prediction", "probability").limit(10))


In [0]:
# Save the results to a permanent table in your catalog
predictions.write.format("delta").mode("overwrite").saveAsTable("final_toy_predictions")

In [0]:
# Convert to Pandas (best for downloading small-to-medium result sets)
# We take a limit (e.g., 10,000 rows) to ensure the download is fast and manageable
df_for_export = predictions.limit(10000).toPandas()

# Define the path in your Volume
export_path = "/Volumes/workspace/default/project/model_results_export.csv"

# Save it
df_for_export.to_csv(export_path, index=False)

print(f"File saved successfully to: {export_path}")

In [0]:
# Ready